# Ejercicio 4 - Analisis temporal de cianobacteria

Este cuaderno resume y visualiza la evolucion temporal del indice de
cianobacteria de los lagos Atitlan y Amatitlan a partir del manifiesto
`data/processed/manifest_indices.csv` generado en el ejercicio anterior.

El cuaderno primero valida ese manifiesto y reporta cuantas de las 22
escenas oficiales ya tienen un raster de cianobacteria calculado. Solo
resume, grafica e interpreta lo que ya esta listo: ninguna fila pendiente
se completa manualmente aqui, y ningun raster se corrige dentro de este
cuaderno.

## 1. Criterio de "pico" (definido antes de mirar los datos)

Una fecha se marca como pico si su promedio de cianobacteria supera en una
desviacion estandar (`PICO_DESVIACIONES` en `src/config.py`) el promedio de
la propia serie del lago. La media y la desviacion de referencia se
calculan solo con fechas de cobertura completa (`quality_flag ==
"calculado"`); las fechas de cobertura parcial se evaluan contra ese mismo
umbral pero no participan en calcularlo.

Este es un umbral descriptivo y repetible, no un modelo de series de
tiempo: cada lago tiene solo 11 observaciones irregulares, insuficientes
para estacionalidad o tendencias estadisticamente robustas.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Abra Jupyter desde la raiz de Laboratorio 4')

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src import analisis_temporal as at
from src.config import PICO_DESVIACIONES

print(f'Raiz del proyecto: {ROOT}')
print(f'Umbral de pico: media + {PICO_DESVIACIONES} desviaciones estandar')


Raiz del proyecto: C:\Users\anna2\Desktop\data-science-lab-4-datos-geoespaciales
Umbral de pico: media + 1.0 desviaciones estandar


## 2. Estado del manifiesto de indices

Cuenta cuantas de las 22 escenas oficiales de cianobacteria estan listas
(raster exportado, `quality_flag` distinto de `pendiente_calculo`) y
cuales siguen pendientes del calculo de indices.

In [2]:
report = at.pending_report()
print(f"Cianobacteria lista: {report['listas']}/{report['total_escenas']} escenas")
print(f"Pendientes: {report['pendientes']}")

pendientes = pd.DataFrame(report['fechas_pendientes'], columns=['lago', 'fecha'])
if not pendientes.empty:
    display(pendientes.groupby('lago').size().rename('fechas_pendientes').to_frame())


Cianobacteria lista: 0/22 escenas
Pendientes: 22


,fechas_pendientes
lago,
amatitlan,11
atitlan,11


## 3. Validacion del manifiesto

No se corrige ninguna fila aqui: si una fila lista le falta unidad, dtype,
pixeles validos o cobertura, la funcion lanza un error explicito y ese
raster debe corregirse en el calculo de indices, no en este cuaderno.

In [3]:
at.validate_indices_manifest_ready()
print('Manifiesto de indices valido para las filas ya calculadas.')


Manifiesto de indices valido para las filas ya calculadas.


## 4. Resumen temporal

Construye una fila por lago y fecha con el promedio, la mediana, la
desviacion estandar y la cobertura valida de cianobacteria, leyendo
directamente los GeoTIFF declarados en el manifiesto. Las escenas todavia
pendientes simplemente no aparecen en la tabla.

In [4]:
resumen = at.build_resumen_temporal()
resumen_df = pd.DataFrame(resumen)

if resumen_df.empty:
    print('Todavia no hay ninguna escena de cianobacteria calculada.')
    print('Esta seccion se completa automaticamente cuando existan raster listos.')
else:
    display(resumen_df)
    path = at.write_resumen_temporal(resumen)
    print(f'resumen_temporal.csv escrito con {len(resumen)} filas en {path}')


Todavia no hay ninguna escena de cianobacteria calculada.
Esta seccion se completa automaticamente cuando existan raster listos.


## 5. Grafico de linea por lago

Una serie por lago, ordenada cronologicamente. Los picos (criterio de la
seccion 1) se marcan con un triangulo y las fechas de cobertura parcial se
marcan con una equis, para que no se lean con la misma confiabilidad que
una escena completa.

In [5]:
if resumen_df.empty:
    print('Sin datos todavia: no se puede graficar.')
else:
    import matplotlib.pyplot as plt

    flagged = pd.DataFrame(at.flag_peaks(resumen))
    flagged['fecha'] = pd.to_datetime(flagged['fecha'])
    flagged = flagged.sort_values(['lago', 'fecha'])

    fig, axes = plt.subplots(len(flagged['lago'].unique()), 1, figsize=(9, 4), squeeze=False, sharex=False)
    for ax, (lago, serie) in zip(axes[:, 0], flagged.groupby('lago')):
        ax.plot(serie['fecha'], serie['cyano_promedio'], marker='o', linewidth=1, label='promedio')

        picos = serie[serie['es_pico']]
        ax.scatter(picos['fecha'], picos['cyano_promedio'], marker='^', color='tab:red', s=90, zorder=3, label='pico')

        parciales = serie[serie['quality_flag'] == 'cobertura_parcial_oficial']
        ax.scatter(parciales['fecha'], parciales['cyano_promedio'], marker='x', color='black', s=90, zorder=3, label='cobertura parcial')

        ax.set_title(lago)
        ax.set_ylabel('cianobacteria (proxy)')
        ax.legend(loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()


Sin datos todavia: no se puede graficar.


## 6. Comparacion conjunta

Los dos lagos usan el mismo indice, la misma formula y la misma unidad
(mismo script de cianobacteria, misma version), por lo que es valido
compararlos en un solo eje.

In [6]:
if resumen_df.empty:
    print('Sin datos todavia: no se puede comparar.')
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    for lago, serie in flagged.groupby('lago'):
        ax.plot(serie['fecha'], serie['cyano_promedio'], marker='o', linewidth=1, label=lago)
    ax.set_ylabel('cianobacteria (proxy)')
    ax.legend()
    plt.tight_layout()
    plt.show()


Sin datos todavia: no se puede comparar.


## 7. Cobertura y confiabilidad

Cobertura valida por fecha; la fila de Amatitlan `2026-02-07` debe
mantenerse marcada como cobertura parcial y no leerse con la misma
confiabilidad que el resto de la serie.

In [7]:
if resumen_df.empty:
    print('Sin datos todavia.')
else:
    display(resumen_df[['lago', 'fecha', 'cobertura_valida_pct', 'quality_flag']])


Sin datos todavia.


## 8. Interpretacion

*(Seccion a completar cuando el resumen tenga datos reales; mantener
siempre separados los tres niveles siguientes.)*

**Observado directamente en los datos:** describir aqui, con base
unicamente en la tabla y las graficas anteriores, si el promedio de
cianobacteria de cada lago sube, baja o fluctua, y en que fechas ocurren
los picos marcados.

**Posible explicacion ambiental:** describir aqui una hipotesis razonable
(por ejemplo estacionalidad de lluvias, temperatura, uso del suelo en la
cuenca) sin presentarla como un hecho comprobado por estos datos.

**Limitaciones:** son 11 observaciones irregulares por lago, no una serie
regular; no se aplica ni se debe interpretar ningun modelo clasico de
series de tiempo ni una estacionalidad robusta. La fecha de cobertura
parcial de Amatitlan (`2026-02-07`, ~57.1% de cobertura valida) no tiene la
misma confiabilidad que las demas y cualquier conclusion que dependa de
ella debe señalarse por separado. El indice de cianobacteria es un proxy
calibrado sobre un dataset simulado, no una medicion de laboratorio.

## 9. Resultado del ejercicio

Cuando el manifiesto de indices tenga las 22 escenas de cianobacteria
calculadas, este cuaderno produce `data/processed/tablas/resumen_temporal.csv`
con una fila por lago y fecha (promedio, mediana, desviacion estandar,
pixeles validos y cobertura), la serie graficada por lago con picos y
cobertura parcial marcados, y la interpretacion de la seccion 8 completa
con las tres partes separadas.